# TriageTune vLLM serving benchmark on a T4

Runs the full serving benchmark from a fresh Colab or Kaggle T4 session and
writes every number to `results/`. The notebook only runs the scripts in
`serving/` and prints the files they produce. It reports nothing of its own.

Before starting:

- Colab: `Runtime` > `Change runtime type` > `T4 GPU`.
- Kaggle: `Settings` > `Accelerator` > `GPU T4 x2`, and turn `Internet` on,
  otherwise the clone and the model download both fail.

Run the cells in order. There is one runtime restart, in step 3, and the cell
immediately after it is the resume point.

No access token is needed anywhere. The base model and the adapter are both
public and every download below passes `token=False`. Do not paste a token
into this notebook.

## Settings

In [ ]:
# Edit BRANCH to benchmark a different branch.
REPO_URL = "https://github.com/vamsi513/triagetune.git"
BRANCH = "vllm-serving"

VLLM_PORT = 8800
HF_PORT = 8765

# vLLM holds this fraction of the card so the FastAPI endpoint can load its
# own copy of the weights on the same GPU at the same time.
SHARED_GPU_FRACTION = "0.45"

import os

if os.path.isdir("/kaggle/working"):
    WORK_ROOT = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK_ROOT = "/content"
else:
    WORK_ROOT = os.getcwd()

REPO_DIR = os.path.join(WORK_ROOT, "triagetune")
print("work root:", WORK_ROOT)
print("repo dir: ", REPO_DIR)
print("branch:   ", BRANCH)

## 1. Check the GPU

In [ ]:
!nvidia-smi

try:
    import torch
except ImportError:
    raise SystemExit(
        "torch is not available in this session. Attach a GPU runtime and rerun."
    )

assert torch.cuda.is_available(), (
    "No CUDA device found. Stop here: vLLM cannot run without an NVIDIA GPU. "
    "Colab: Runtime > Change runtime type > T4 GPU. "
    "Kaggle: Settings > Accelerator > GPU T4 x2."
)

name = torch.cuda.get_device_name(0)
capability = ".".join(str(part) for part in torch.cuda.get_device_capability(0))
print("GPU:        ", name)
print("capability: ", capability)
print("total bytes:", torch.cuda.get_device_properties(0).total_memory)
if capability == "7.5":
    print("sm_75 has no bfloat16; the launch script uses --dtype half.")

## 2. Clone the repo and check out the branch

In [ ]:
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--quiet"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
os.chdir(REPO_DIR)

print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

## 3. Install dependencies

In [ ]:
!pip install -q -r serving/requirements-vllm.txt -r requirements-serving.txt

## **RESTART THE RUNTIME NOW**

vLLM pins its own torch build and the install above replaced the torch this
session already had loaded. The old one stays in memory until the process is
restarted.

- Colab: `Runtime` > `Restart session`.
- Kaggle: `Run` > `Restart session`.

Then continue from the next cell, not from the top. The restart clears every
Python variable, so the next cell sets them again.

### Resume point after the restart

In [ ]:
# The restart cleared all variables, so set them again here. Keep these values
# in step with the Settings cell above if you changed it.
import os
import subprocess

REPO_URL = "https://github.com/vamsi513/triagetune.git"
BRANCH = "vllm-serving"
VLLM_PORT = 8800
HF_PORT = 8765
SHARED_GPU_FRACTION = "0.45"
FULL_GPU_FRACTION = "0.90"

if os.path.isdir("/kaggle/working"):
    WORK_ROOT = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK_ROOT = "/content"
else:
    WORK_ROOT = os.getcwd()

REPO_DIR = os.path.join(WORK_ROOT, "triagetune")
os.chdir(REPO_DIR)
os.makedirs("results", exist_ok=True)

ADAPTER_DIR = "artifacts/lora_adapter"
CACHE_DIR = ".cache/huggingface"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BASE_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
ADAPTER_REPO = "Vamsi513/triagetune-banking77-lora"

VLLM_LOG = "results/serving_vllm_server.log"
VLLM_PID_FILE = "results/serving_vllm_server.pid"
HF_LOG = "results/serving_huggingface_server.log"
HF_PID_FILE = "results/serving_huggingface_server.pid"


def tail(path, lines=60):
    """Print the last lines of a log, or say that it is missing."""
    if not os.path.isfile(path):
        print(f"no log at {path}")
        return
    with open(path, encoding="utf-8", errors="replace") as handle:
        content = handle.read().splitlines()
    print(f"--- last {min(lines, len(content))} lines of {path} ---")
    for line in content[-lines:]:
        print(line)


def wait_for_health(port, timeout_seconds, path="/health"):
    """Poll an endpoint until it answers, or time out. Returns True on success."""
    import time
    import urllib.error
    import urllib.request

    url = f"http://127.0.0.1:{port}{path}"
    deadline = time.time() + timeout_seconds
    attempt = 0
    while time.time() < deadline:
        attempt += 1
        try:
            with urllib.request.urlopen(url, timeout=10) as response:
                if response.status == 200:
                    waited = int(timeout_seconds - (deadline - time.time()))
                    print(f"{url} healthy after about {waited} seconds")
                    return True
        except (urllib.error.URLError, OSError):
            pass
        if attempt % 6 == 0:
            print(f"still waiting for {url} ({int(deadline - time.time())}s left)")
        time.sleep(5)
    print(f"timed out waiting for {url}")
    return False


import torch

print("cwd:          ", os.getcwd())
print("torch:        ", torch.__version__)
print("cuda runtime: ", torch.version.cuda)
print("cuda device:  ", torch.cuda.get_device_name(0))

## 4. Download the adapter and the base model

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=ADAPTER_REPO,
    local_dir=ADAPTER_DIR,
    token=False,
)
snapshot_download(
    repo_id=BASE_MODEL,
    revision=BASE_REVISION,
    cache_dir=CACHE_DIR,
    token=False,
)

adapter_weights = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
assert os.path.isfile(adapter_weights), f"adapter weights missing at {adapter_weights}"
print("adapter weight bytes:", os.path.getsize(adapter_weights))
print("adapter files:", sorted(os.listdir(ADAPTER_DIR)))

## 5. Start the vLLM server

In [ ]:
# serve_vllm.sh records results/serving_env.json, then backgrounds the server
# and waits for it to report healthy. start_new_session keeps the server alive
# after this cell finishes.
environment = dict(os.environ)
environment["VLLM_GPU_FRACTION"] = SHARED_GPU_FRACTION

launch = subprocess.run(
    ["bash", "serving/serve_vllm.sh", "--background"],
    env=environment,
    start_new_session=True,
    capture_output=True,
    text=True,
    timeout=1800,
)
print(launch.stdout)
if launch.stderr.strip():
    print(launch.stderr)

if not wait_for_health(VLLM_PORT, timeout_seconds=600):
    tail(VLLM_LOG, 60)
    raise SystemExit(
        "vLLM did not become healthy within 10 minutes. Read the log above, "
        "then see the fallback cell at the end of this notebook."
    )

print("vLLM is serving on port", VLLM_PORT)

## 6. Start the Hugging Face FastAPI endpoint

In [ ]:
# TRIAGETUNE_ENABLE_PROVISIONAL_ROUTING=1 is the opt-in switch from the README.
# Without it /classify returns HTTP 503 and the benchmark has nothing to call.
environment = dict(os.environ)
environment["TRIAGETUNE_ENABLE_PROVISIONAL_ROUTING"] = "1"

with open(HF_LOG, "w", encoding="utf-8") as log_handle:
    hf_server = subprocess.Popen(
        [
            "python3", "-m", "uvicorn", "api.main:app",
            "--host", "127.0.0.1", "--port", str(HF_PORT),
        ],
        env=environment,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

with open(HF_PID_FILE, "w", encoding="utf-8") as handle:
    handle.write(f"{hf_server.pid}\n")
print("FastAPI pid", hf_server.pid)

# The adapter loads at startup, so /health only answers once the weights are
# resident on the GPU.
if not wait_for_health(HF_PORT, timeout_seconds=300):
    tail(HF_LOG, 60)
    raise SystemExit(
        "The FastAPI endpoint did not become healthy within 5 minutes. "
        "Read the log above."
    )

print("FastAPI endpoint is serving on port", HF_PORT)

## 7. Score the reserved test set

In [ ]:
# All 3,080 reserved test requests at temperature 0. This is the long step.
!python3 serving/eval_vllm.py

import json

with open("results/serving_accuracy.json", encoding="utf-8") as handle:
    accuracy_report = json.load(handle)

print(json.dumps(accuracy_report["metrics"], indent=2)[:400])
print(json.dumps(accuracy_report["structured_output"], indent=2))
print(json.dumps(accuracy_report["request_health"], indent=2))

### Accuracy gap against the recorded run

In [ ]:
# The README records 80.49% for the saved adapter. eval_vllm.py already
# computed the gap against reports/lora_adapter_test.json; this cell reads
# that field and flags it.
comparison = accuracy_report["comparison_with_recorded_huggingface_run"]

recorded = comparison["reference_accuracy"]
measured = comparison["vllm_accuracy"]
gap_points = comparison["accuracy_delta_points"]
tolerance = comparison["accuracy_tolerance_points"]

print(f"recorded accuracy : {recorded * 100:.2f}%")
print(f"vLLM accuracy     : {measured * 100:.2f}%")
print(f"gap               : {gap_points:+.2f} points (tolerance {tolerance})")
print(f"disagreements     : {comparison['disagreements']} of "
      f"{comparison['compared_examples']} compared examples")

if comparison["accuracy_outside_tolerance"]:
    print()
    print("WARNING: the gap is wider than the tolerance.")
    print("Investigate before quoting this number anywhere. In order:")
    print("  1. Chat template. Rerun with --mode chat and compare, since the")
    print("     prompt text itself is already hash checked before any request.")
    print("  2. Tokenization. Compare the transformers version in")
    print("     results/serving_env.json against the recorded run.")
    print("  3. Dtype. The recorded run used float16; confirm --dtype half.")
    print("  See serving/profile_notes.md for the full checklist.")
else:
    print()
    print("The gap is within tolerance.")

for finding in accuracy_report["findings"]:
    print("FINDING:", finding)

## 8. Load test both backends

In [ ]:
# The same 300 seeded requests go to both backends at concurrency 1, 4, 16
# and 32, with 20 warmup requests per run discarded.
!python3 serving/bench.py

from IPython.display import Image, Markdown, display

with open("results/serving_bench.md", encoding="utf-8") as handle:
    display(Markdown(handle.read()))

if os.path.isfile("results/serving_bench.png"):
    display(Image(filename="results/serving_bench.png"))
else:
    print("no chart at results/serving_bench.png")

## 9. Optional: vLLM alone at full memory

Optional. Skip it if the session is short on time.

Step 5 gave vLLM only 45% of the card so both servers could be resident at
once. That shrinks the KV cache and limits how many sequences vLLM can batch,
so the shared run understates what vLLM does at concurrency 16 and 32. This
cell stops the FastAPI endpoint, restarts vLLM with the full card, and runs the
load test against vLLM alone.

The output goes to `results/serving_bench_vllm_only.*`, so the shared run in
`results/serving_bench.*` is left intact. The two are not comparable with each
other: only the shared run measured both backends under the same conditions.

In [ ]:
import signal
import time

# Stop the FastAPI endpoint to free its share of the GPU.
if os.path.isfile(HF_PID_FILE):
    with open(HF_PID_FILE, encoding="utf-8") as handle:
        pid = int(handle.read().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print("sent SIGTERM to FastAPI pid", pid)
    except ProcessLookupError:
        print("FastAPI pid", pid, "was already gone")

# Stop the shared vLLM server.
if os.path.isfile(VLLM_PID_FILE):
    with open(VLLM_PID_FILE, encoding="utf-8") as handle:
        pid = int(handle.read().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print("sent SIGTERM to vLLM pid", pid)
    except ProcessLookupError:
        print("vLLM pid", pid, "was already gone")

time.sleep(30)

# Restart vLLM with the whole card. This overwrites results/serving_env.json,
# which is correct: it then describes the run that produced the files below.
environment = dict(os.environ)
environment["VLLM_GPU_FRACTION"] = FULL_GPU_FRACTION

launch = subprocess.run(
    ["bash", "serving/serve_vllm.sh", "--background"],
    env=environment,
    start_new_session=True,
    capture_output=True,
    text=True,
    timeout=1800,
)
print(launch.stdout)
if launch.stderr.strip():
    print(launch.stderr)

if not wait_for_health(VLLM_PORT, timeout_seconds=600):
    tail(VLLM_LOG, 60)
    raise SystemExit("vLLM did not come back up at the full memory fraction.")

subprocess.run(
    [
        "python3", "serving/bench.py", "--backends", "vllm",
        "--json-file", "results/serving_bench_vllm_only.json",
        "--markdown-file", "results/serving_bench_vllm_only.md",
        "--chart-file", "results/serving_bench_vllm_only.png",
    ],
    check=True,
)

# Bug 3 guard: import here too so this optional cell does not depend on step 8.
from IPython.display import Markdown, display

with open("results/serving_bench_vllm_only.md", encoding="utf-8") as handle:
    display(Markdown(handle.read()))

## 10. Package the results

In [ ]:
import json
import shutil

# The hardware these numbers belong to. Keep it with them.
with open("results/serving_env.json", encoding="utf-8") as handle:
    print(json.dumps(json.load(handle), indent=2))

print()
print("files in results/:")
for name in sorted(os.listdir("results")):
    print(f"  {os.path.getsize(os.path.join('results', name)):>10} bytes  {name}")

archive = shutil.make_archive(
    base_name=os.path.join(WORK_ROOT, "results"),
    format="zip",
    root_dir=os.getcwd(),
    base_dir="results",
)
print()
print("archive:", archive, os.path.getsize(archive), "bytes")

try:
    from google.colab import files
except ImportError:
    print()
    print("Kaggle or local: the archive is at the path above.")
    print("On Kaggle, anything under /kaggle/working appears in the Output")
    print("panel on the right once the session is committed or saved.")
else:
    files.download(archive)

## If vLLM will not start on the T4

Check `results/serving_vllm_server.log` first, then try these in order,
rerunning step 5 after each change:

1. `VLLM_EXTRA_ARGS="--enforce-eager"` skips CUDA graph capture. It lowers
   throughput but cuts startup memory, and it is the usual fix for an
   out of memory failure during startup.
2. Lower `VLLM_GPU_FRACTION` further, for example `0.35`.
3. `VLLM_MAX_MODEL_LEN=768`. The recorded prompts are around 450 tokens, with
   a maximum of 532 in training, so 768 still covers the test set and leaves
   more room for the KV cache.
4. If the pinned `vllm==0.32.0` itself fails on sm_75, try `vllm==0.31.0`.
   I have not run either version on a T4, so I cannot say which works; record
   whichever one you used in `results/serving_env.json`, which the launch
   script writes from the installed packages.

To set one of these for a rerun, change the environment dictionary in step 5,
for example:

```python
environment["VLLM_EXTRA_ARGS"] = "--enforce-eager"
```

A free T4 is shared, throttled hardware, and the session can be reclaimed mid
run. One GPU is one data point: these numbers do not carry over to an A10G, an
L4, an A100, or to CPU serving.